# Frameworks 02 - AWS Strands

Objetivo: ejecutar strands.Agent real con Tools mixtas, hooks, structured
output, sync/async y MCP local por stdio y Streamable HTTP.


**Lugar en el modelo:** AWS Strands es el Framework que controla lifecycle, hooks, Tools y MCP; Bedrock es sólo uno de los Providers posibles.

**Evidencia exigida:** el SDK real debe ejecutar sync/async, structured output, hooks y ambos transports MCP, todos con evidencia verificable.

**Límite de la evidencia:** `python-runtime` hace determinista el modelo offline; Strands no reemplaza el plan de ejecución de un System ni convierte cualquier pipeline Python en un agent loop.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_STRANDS_LIVE | 0 | Cambia Python por Provider auto. |
| Provider offline | python-runtime | Modelo scripted determinista. |
| Framework | strands | Lifecycle, hooks, Tools y MCP reales. |


## 1) Provider x Framework y lifecycle


In [ ]:
import os

from pydantic import BaseModel
from strands import tool as strands_tool
from strands.hooks import AfterInvocationEvent

import agentic_systems as toolkit

RUN_LIVE = os.getenv("RUN_STRANDS_LIVE", "0").strip().lower() in {"1", "true", "yes"}
runtime = (
    toolkit.runtime(
        provider="auto",
        provider_priority=["openai-runtime", "vllm-runtime", "bedrock-runtime"],
    )
    if RUN_LIVE
    else toolkit.runtime(provider="python-runtime")
)
profile = toolkit.integrations.framework_profile("strands")
runtime_description = runtime.describe()
toolkit.show_json(
    {"runtime": runtime_description, "profile": profile.to_dict()},
    title="Provider x Strands",
)


## 2) Tools mixtas, hook, structured output y sync/async


In [ ]:
class PublicEvidence(BaseModel):
    symbol: str
    is_public: bool

hook_events = []

def record_after_invocation(event: AfterInvocationEvent):
    hook_events.append(type(event).__name__)

@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@strands_tool
def native_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

framework = toolkit.framework(
    "strands",
    agent_kwargs={"tools": [native_public_api], "hooks": [record_after_invocation]},
    run_kwargs={"structured_output_model": PublicEvidence},
)
agent = toolkit.agent(
    name="strands_inspector",
    instructions="Usa la Tool solicitada y devuelve PublicEvidence.",
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    policy=toolkit.RunPolicy(max_turns=4, max_tool_calls=2),
)
agent.prepare()
sync_result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "framework"}},
    mode="eval",
)
async_result = await agent.arun(
    {"tool": "native_public_api", "input": {"symbol": "Skill"}},
    mode="eval",
)
assert sync_result.ok and async_result.ok
assert sync_result.engine == async_result.engine == runtime_description["selected_provider"]
assert sync_result.meta["framework_adapter"] == "strands"
assert async_result.meta["framework_adapter"] == "strands"
assert hook_events == ["AfterInvocationEvent", "AfterInvocationEvent"]
toolkit.human_result(sync_result, title="Strands RunResult", show_lineage=True)
toolkit.show_json(
    {
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(sync_result.native_result).__name__,
        "hook_events": hook_events,
        "async_data": async_result.data,
    },
    title="Strands lifecycle evidence",
)


## 3) MCP local por stdio y Streamable HTTP


In [ ]:
from collections.abc import AsyncIterator, Callable
from contextlib import AbstractAsyncContextManager, asynccontextmanager
from functools import partial
from pathlib import Path
import os
import socket
import subprocess
import sys
import time

from mcp.client.stdio import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient

def locate_repo_file(relative: str) -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = (root / relative).resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {relative!r} from kernel cwd {Path.cwd()}."
    )

SERVER = locate_repo_file("tutorials/frameworks/mcp_echo_server.py")

def free_port() -> int:
    with socket.socket() as listener:
        listener.bind(("127.0.0.1", 0))
        return int(listener.getsockname()[1])

def wait_for_port(process, port: int) -> None:
    deadline = time.monotonic() + 15
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("The local MCP server exited early.")
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.05)
    raise TimeoutError("The local MCP server did not start.")

@asynccontextmanager
async def stdio_transport() -> AsyncIterator:
    parameters = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER), "--transport", "stdio"],
    )
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(parameters, errlog=errlog) as transport:
            yield transport

def run_mcp_transport(transport: str):
    process = None
    transport_factory: Callable[[], AbstractAsyncContextManager]
    if transport == "stdio":
        transport_factory = stdio_transport
    else:
        port = free_port()
        process = subprocess.Popen(
            [sys.executable, str(SERVER), "--transport", "streamable-http", "--port", str(port)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        wait_for_port(process, port)
        transport_factory = partial(streamable_http_client, f"http://127.0.0.1:{port}/mcp")

    client = MCPClient(transport_factory)
    try:
        mcp_agent = toolkit.agent(
            name=f"strands_mcp_{transport}",
            instructions="Ejecuta la Tool MCP solicitada.",
            runtime=toolkit.runtime(provider="python-runtime"),
            framework=toolkit.framework("strands", agent_kwargs={"tools": [client]}),
        )
        return mcp_agent.run(
            {"tool": "echo", "input": {"value": "verified"}},
            mode="eval",
        )
    finally:
        client.stop(None, None, None)
        if process is not None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)

mcp_results = {
    transport: run_mcp_transport(transport)
    for transport in ("stdio", "streamable-http")
}
assert all(result.ok for result in mcp_results.values())
toolkit.show_json(
    {
        transport: {
            "data": result.data,
            "tool_events": [event.name for event in result.tool_events],
        }
        for transport, result in mcp_results.items()
    },
    title="Native Strands MCP",
)


## 4) API realmente ejercitada


In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.framework", "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.RunPolicy",
    "toolkit.human_result", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Strands API coverage")


## Resultado e interpretacion

Strands controla lifecycle, hooks, Tools, structured output y ambos transports
MCP. Agentic Systems conserva Provider, policy y normalizacion a RunResult.
